# Article retrieval

Builds the reference article resources used throughout the pipeline: the full text of every Civil Code article, the list of article numbers, and the 2016 old/new numbering equivalences (JO 11/02/2016 reform).

## Fetch article texts

Retrieves the full text of every Civil Code article from the **Legifrance** API (sandbox, `pylegifrance`). Progress is cached so the fetch can resume; the merged result is the shipped `DATA/inputs/civil_code_articles.json`.

In [ ]:
import os
import time
import json
from pylegifrance import LegiHandler, recherche_CODE
from pylegifrance.process.processors import GetArticleIdError

def init_client_sandbox() -> LegiHandler:
    client = LegiHandler()
    client.token_url = "https://sandbox-oauth.piste.gouv.fr/api/oauth/token"
    client.api_url = "https://sandbox-api.piste.gouv.fr/dila/legifrance/lf-engine-app/"
    client.client_id = None
    client.client_secret = None
    client.set_api_keys(
        legifrance_api_key=os.environ["LEGIFRANCE_API_KEY"],
        legifrance_api_secret=os.environ["LEGIFRANCE_API_SECRET"]
    )
    return client

def fetch_articles_texts(article_nums: list[str], save_file="artifacts/reference/progression.json", time_limit=3300) -> dict[str, str]:
    """
    For each article number, retrieve its full text from the Civil Code.
    Saves progress periodically to allow resuming.
    """
    # Load existing progress if available
    try:
        with open(save_file, "r", encoding="utf-8") as f:
            results = json.load(f)
    except FileNotFoundError:
        results = {}

    init_client_sandbox()
    start_time = time.time()

    for num in article_nums:
        if num in results:
            continue  # already processed

        if (time.time() - start_time) > time_limit:
            print(f"[INFO] Time limit reached after {len(results)} articles. Saving and stopping.")
            break

        try:
            df = recherche_CODE(
                code_name="Code civil",
                search=num,
                formatter=True
            )
        except GetArticleIdError:
            continue  # article not found
        except Exception as e:
            print(f"[ERROR] on {num}: {e}")
            continue

        # Extract content
        if isinstance(df, dict):
            texte = df.get("texte") or df.get("content")
        else:
            if df.empty:
                continue
            texte = df.iloc[0].get("texte") or df.iloc[0].get("content")

        if texte:
            results[num] = texte.strip()

        # Save every 200 articles
        if len(results) % 200 == 0:
            with open(save_file, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            print(f"[INFO] Saved after {len(results)} articles.")

    # Final save
    with open(save_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("[INFO] Done or time limit reached. Results saved.")
    return results



In [ ]:
import json, os

os.makedirs("artifacts/reference", exist_ok=True)  # not shipped — regenerated by this step (see DATA.md)

# Complete list of Civil Code article numbers to retrieve
articles = ['1', '2', '3', '4', '5', '6', '6-1', '6-2', '7', '8', '9', '9-1', '10', '11', '14', '15', '16', '16-1', '16-10', '16-11', '16-12', '16-13', '16-14', '16-2', '16-3', '16-4', '16-5', '16-6', '16-7', '16-8', '16-9', '17', '17-1', '17-10', '17-11', '17-12', '17-2', '17-3', '17-4', '17-5', '17-6', '17-7', '17-8', '17-9', '18', '18-1', '19', '19-1', '19-2', '19-3', '19-4', '20', '20-1', '20-2', '20-3', '20-4', '20-5', '21', '21-1', '21-10', '21-11', '21-12', '21-13', '21-14', '21-15', '21-16', '21-17', '21-18', '21-19', '21-2', '21-20', '21-21', '21-22', '21-23', '21-24', '21-25', '21-26', '21-27', '21-28', '21-29', '21-3', '21-4', '21-5', '21-6', '21-7', '21-8', '21-9', '22', '22-1', '22-2', '22-3', '23', '23-1', '23-2', '23-3', '23-4', '23-5', '23-6', '23-7', '23-8', '23-9', '24', '24-1', '24-2', '24-3', '25', '25-1', '26', '26-1', '26-2', '26-3', '26-4', '26-5', '27', '27-1', '27-2', '27-3', '28', '28-1', '29', '29-1', '29-2', '29-3', '29-4', '29-5', '30', '30-1', '30-2', '30-3', '30-4', '31', '31-1', '31-2', '31-3', '32', '32-1', '32-2', '32-3', '32-4', '32-5', '33', '33-1', '33-2', '34', '34-1', '35', '36', '37', '38', '39', '40', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '57-1', '58', '59', '60', '61', '61-1', '61-2', '61-3', '61-4', '61-5', '61-6', '61-7', '61-8', '62', '62-1', '63', '64', '65', '66', '67', '68', '69', '70', '71', '73', '74', '74-1', '75', '76', '78', '79', '79-1', '80', '81', '82', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '95', '96', '96-1', '96-2', '97', '98', '98-1', '98-2', '98-3', '98-4', '99', '99-1', '99-2', '100', '101', '101-1', '101-2', '102', '103', '104', '105', '106', '107', '108', '108-1', '108-2', '108-3', '109', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '143', '144', '145', '146', '146-1', '147', '148', '149', '150', '151', '154', '155', '156', '157', '159', '160', '161', '162', '163', '164', '165', '166', '169', '171', '171-1', '171-2', '171-3', '171-4', '171-5', '171-6', '171-7', '171-8', '171-9', '172', '173', '174', '175', '175-1', '175-2', '176', '177', '178', '179', '180', '181', '182', '183', '184', '187', '188', '189', '190', '191', '192', '193', '194', '195', '196', '197', '198', '199', '200', '201', '202', '202-1', '202-2', '203', '204', '205', '206', '207', '208', '209', '210', '211', '212', '213', '214', '215', '216', '217', '218', '219', '220', '220-1', '220-2', '220-3', '221', '222', '223', '225', '225-1', '226', '227', '229', '229-1', '229-2', '229-3', '229-4', '230', '232', '233', '234', '237', '238', '242', '244', '245', '245-1', '246', '247', '247-1', '247-2', '248', '249', '249-2', '249-3', '249-4', '250', '250-1', '250-2', '250-3', '251', '252', '253', '254', '255', '256', '259', '259-1', '259-2', '259-3', '260', '262', '262-1', '262-2', '263', '264', '265', '265-1', '265-2', '266', '267', '268', '270', '271', '272', '274', '275', '275-1', '276', '276-1', '276-3', '276-4', '277', '278', '279', '279-1', '280', '280-1', '280-2', '281', '285-1', '286', '296', '297', '297-1', '298', '299', '300', '301', '302', '303', '304', '305', '306', '307', '308', '309', '310-1', '310-2', '310-3', '311', '311-1', '311-14', '311-15', '311-17', '311-2', '311-21', '311-22', '311-23', '311-24', '311-25', '312', '313', '314', '315', '316', '316-1', '316-2', '316-3', '316-4', '316-5', '317', '318', '318-1', '319', '320', '321', '322', '323', '324', '325', '326', '327', '328', '329', '330', '331', '332', '333', '334', '335', '336', '336-1', '337', '342', '342-10', '342-11', '342-12', '342-13', '342-2', '342-4', '342-5', '342-6', '342-7', '342-8', '342-9', '343', '343-1', '344', '345', '345-1', '345-2', '346', '347', '348', '348-1', '348-2', '348-3', '348-4', '348-5', '348-6', '348-7', '349', '350', '351', '352', '352-1', '352-2', '353', '353-1', '353-2', '354', '355', '356', '357', '358', '359', '360', '361', '362', '363', '363-1', '364', '365', '366', '367', '368', '369', '369-1', '370', '370-1', '370-2', '370-3', '370-4', '370-5', '371', '371-1', '371-2', '371-3', '371-4', '371-5', '371-6', '372', '372-1', '372-2', '373', '373-1', '373-2', '373-3', '373-4', '373-5', '374-1', '374-2', '375', '375-1', '375-2', '375-3', '375-4', '375-5', '375-6', '375-7', '375-8', '375-9', '376', '376-1', '377', '377-1', '377-2', '377-3', '378', '378-1', '378-2', '379', '379-1', '380', '380-1', '381', '381-1', '381-2', '382', '382-1', '383', '384', '385', '386', '386-1', '386-2', '386-3', '386-4', '387', '387-1', '387-2', '387-3', '387-4', '387-5', '387-6', '388', '388-1', '388-2', '390', '391', '392', '393', '394', '395', '396', '397', '398', '399', '400', '401', '402', '403', '404', '405', '406', '407', '408', '408-1', '409', '410', '411', '411-1', '412', '413', '413-1', '413-2', '413-3', '413-4', '413-5', '413-6', '413-7', '413-8', '414', '414-1', '414-2', '414-3', '415', '416', '417', '418', '419', '420', '421', '422', '423', '424', '425', '426', '427', '428', '429', '430', '431', '432', '433', '434', '435', '436', '437', '438', '439', '440', '441', '442', '443', '444', '445', '446', '447', '448', '449', '450', '451', '452', '453', '454', '455', '456', '457', '457-1', '458', '459', '459-1', '459-2', '460', '461', '462', '463', '464', '465', '466', '467', '468', '469', '470', '471', '472', '473', '474', '475', '476', '477', '477-1', '478', '479', '480', '481', '482', '483', '484', '485', '486', '487', '488', '489', '490', '491', '492', '492-1', '493', '494', '494-1', '494-10', '494-11', '494-12', '494-2', '494-3', '494-4', '494-5', '494-6', '494-7', '494-8', '494-9', '495', '495-1', '495-2', '495-3', '495-4', '495-5', '495-6', '495-7', '495-8', '495-9', '496', '497', '498', '499', '500', '501', '502', '503', '504', '505', '506', '507', '507-1', '507-2', '508', '509', '510', '511', '512', '513', '513-1', '514', '515', '515-1', '515-10', '515-11', '515-12', '515-13', '515-14', '515-2', '515-3', '515-4', '515-5', '515-6', '515-7', '515-8', '515-9', '516', '517', '518', '519', '520', '521', '522', '523', '524', '525', '526', '527', '528', '529', '530', '531', '532', '533', '534', '535', '536', '537', '539', '542', '543', '544', '545', '546', '547', '548', '549', '550', '551', '552', '553', '554', '555', '556', '557', '558', '559', '560', '561', '562', '563', '564', '565', '566', '567', '568', '569', '570', '571', '572', '573', '574', '575', '576', '577', '578', '579', '580', '581', '582', '583', '584', '585', '586', '587', '588', '589', '590', '591', '592', '593', '594', '595', '596', '597', '598', '599', '600', '601', '602', '603', '604', '605', '606', '607', '608', '609', '610', '611', '612', '613', '614', '615', '616', '617', '618', '619', '620', '621', '622', '623', '624', '625', '626', '627', '628', '629', '630', '631', '632', '633', '634', '635', '636', '637', '638', '639', '640', '641', '642', '643', '644', '645', '646', '647', '648', '649', '650', '651', '652', '653', '654', '655', '656', '657', '658', '659', '660', '661', '662', '663', '665', '666', '667', '668', '669', '670', '671', '672', '673', '674', '675', '676', '677', '678', '679', '680', '681', '682', '683', '684', '685', '685-1', '686', '687', '688', '689', '690', '691', '692', '693', '694', '695', '696', '697', '698', '699', '700', '701', '702', '703', '704', '705', '706', '707', '708', '709', '710', '710-1', '711', '712', '713', '714', '715', '716', '717', '720', '721', '722', '724', '724-1', '725', '725-1', '726', '727', '727-1', '728', '729', '729-1', '730', '730-1', '730-2', '730-3', '730-4', '730-5', '731', '732', '733', '734', '735', '736', '737', '738', '738-1', '738-2', '739', '740', '741', '742', '743', '744', '745', '746', '747', '748', '749', '750', '751', '752', '752-1', '752-2', '753', '754', '755', '756', '757', '757-1', '757-2', '757-3', '758', '758-1', '758-2', '758-3', '758-4', '758-5', '758-6', '759', '759-1', '760', '761', '762', '763', '764', '765', '765-1', '765-2', '766', '767', '768', '769', '770', '771', '772', '773', '774', '775', '776', '777', '778', '779', '780', '781', '782', '783', '784', '785', '786', '787', '788', '789', '790', '791', '792', '792-1', '792-2', '793', '794', '795', '796', '797', '798', '799', '800', '801', '802', '803', '804', '805', '806', '807', '808', '809', '809-1', '809-2', '809-3', '810', '810-1', '810-10', '810-11', '810-12', '810-2', '810-3', '810-4', '810-5', '810-6', '810-7', '810-8', '810-9', '811', '811-1', '811-2', '811-3', '812', '812-1', '812-2', '812-3', '812-4', '812-5', '812-6', '812-7', '813', '813-1', '813-2', '813-3', '813-4', '813-5', '813-6', '813-7', '813-8', '813-9', '814', '814-1', '815', '815-1', '815-10', '815-11', '815-12', '815-13', '815-14', '815-15', '815-16', '815-17', '815-18', '815-2', '815-3', '815-4', '815-5', '815-6', '815-7', '815-8', '815-9', '816', '817', '818', '819', '820', '821', '821-1', '822', '823', '824', '825', '826', '827', '828', '829', '830', '831', '831-1', '831-2', '831-3', '832', '832-1', '832-2', '832-3', '832-4', '833', '834', '835', '836', '837', '838', '839', '840', '840-1', '841', '841-1', '842', '843', '844', '845', '846', '847', '848', '849', '850', '851', '852', '853', '854', '855', '856', '857', '858', '859', '860', '860-1', '861', '862', '863', '864', '865', '866', '867', '870', '871', '872', '873', '874', '875', '876', '877', '878', '879', '880', '881', '882', '883', '884', '885', '886', '887', '887-1', '888', '889', '890', '891', '892', '893', '894', '895', '896', '898', '899', '900', '900-1', '900-2', '900-3', '900-4', '900-5', '900-6', '900-7', '900-8', '901', '902', '903', '904', '906', '907', '909', '910', '910-1', '911', '912', '913', '913-1', '914-1', '916', '917', '918', '919', '919-1', '919-2', '920', '921', '922', '923', '924', '924-1', '924-2', '924-3', '924-4', '926', '927', '928', '929', '930', '930-1', '930-2', '930-3', '930-4', '930-5', '931', '931-1', '932', '933', '935', '936', '937', '938', '939', '940', '941', '942', '943', '944', '945', '946', '947', '948', '949', '950', '951', '952', '953', '954', '955', '956', '957', '958', '959', '960', '961', '962', '963', '964', '965', '966', '967', '968', '969', '970', '971', '972', '973', '974', '975', '976', '977', '978', '979', '980', '981', '982', '983', '984', '985', '986', '987', '988', '989', '990', '991', '992', '993', '994', '995', '996', '997', '998', '999', '1000', '1001', '1002', '1002-1', '1003', '1004', '1005', '1006', '1007', '1009', '1010', '1011', '1012', '1013', '1014', '1015', '1016', '1017', '1018', '1019', '1020', '1021', '1022', '1023', '1024', '1025', '1026', '1027', '1028', '1029', '1030', '1030-1', '1030-2', '1031', '1032', '1033', '1033-1', '1034', '1035', '1036', '1037', '1038', '1039', '1040', '1041', '1042', '1043', '1044', '1045', '1046', '1047', '1048', '1049', '1050', '1051', '1052', '1053', '1054', '1055', '1056', '1057', '1058', '1059', '1060', '1061', '1075', '1075-1', '1075-2', '1075-3', '1075-4', '1075-5', '1076', '1076-1', '1077', '1077-1', '1077-2', '1078', '1078-1', '1078-10', '1078-2', '1078-3', '1078-4', '1078-5', '1078-6', '1078-7', '1078-8', '1078-9', '1079', '1080', '1081', '1082', '1083', '1084', '1085', '1086', '1087', '1088', '1089', '1090', '1091', '1092', '1093', '1094', '1094-1', '1094-3', '1095', '1096', '1098', '1099', '1099-1', '1100', '1100-1', '1100-2', '1101', '1102', '1103', '1104', '1105', '1106', '1107', '1108', '1109', '1110', '1111', '1111-1', '1112', '1112-1', '1112-2', '1113', '1114', '1115', '1116', '1117', '1118', '1119', '1120', '1121', '1122', '1123', '1124', '1125', '1126', '1127', '1127-1', '1127-2', '1127-3', '1127-4', '1128', '1129', '1130', '1131', '1132', '1133', '1134', '1135', '1136', '1137', '1138', '1139', '1140', '1141', '1142', '1143', '1144', '1145', '1146', '1147', '1148', '1149', '1150', '1151', '1152', '1153', '1154', '1155', '1156', '1157', '1158', '1159', '1160', '1161', '1162', '1163', '1164', '1165', '1166', '1167', '1168', '1169', '1170', '1171', '1172', '1173', '1174', '1175', '1176', '1177', '1178', '1179', '1180', '1181', '1182', '1183', '1184', '1185', '1186', '1187', '1188', '1189', '1190', '1191', '1192', '1193', '1194', '1195', '1196', '1197', '1198', '1199', '1200', '1201', '1202', '1203', '1204', '1205', '1206', '1207', '1208', '1209', '1210', '1211', '1212', '1213', '1214', '1215', '1216', '1216-1', '1216-2', '1216-3', '1217', '1218', '1219', '1220', '1221', '1222', '1223', '1224', '1225', '1226', '1227', '1228', '1229', '1230', '1231', '1231-1', '1231-2', '1231-3', '1231-4', '1231-5', '1231-6', '1231-7', '1240', '1241', '1242', '1243', '1244', '1245', '1245-1', '1245-10', '1245-11', '1245-12', '1245-13', '1245-14', '1245-15', '1245-16', '1245-17', '1245-2', '1245-3', '1245-4', '1245-5', '1245-6', '1245-7', '1245-8', '1245-9', '1246', '1247', '1248', '1249', '1250', '1251', '1252', '1253', '1300', '1301', '1301-1', '1301-2', '1301-3', '1301-4', '1301-5', '1302', '1302-1', '1302-2', '1302-3', '1303', '1303-1', '1303-2', '1303-3', '1303-4', '1304', '1304-1', '1304-2', '1304-3', '1304-4', '1304-5', '1304-6', '1304-7', '1305', '1305-1', '1305-2', '1305-3', '1305-4', '1305-5', '1306', '1307', '1307-1', '1307-2', '1307-3', '1307-4', '1307-5', '1308', '1309', '1310', '1311', '1312', '1313', '1314', '1315', '1316', '1317', '1318', '1319', '1320', '1321', '1322', '1323', '1324', '1325', '1326', '1327', '1327-1', '1327-2', '1328', '1328-1', '1329', '1330', '1331', '1332', '1333', '1334', '1335', '1336', '1337', '1338', '1339', '1340', '1341', '1341-1', '1341-2', '1341-3', '1342', '1342-1', '1342-10', '1342-2', '1342-3', '1342-4', '1342-5', '1342-6', '1342-7', '1342-8', '1342-9', '1343', '1343-1', '1343-2', '1343-3', '1343-4', '1343-5', '1344', '1344-1', '1344-2', '1345', '1345-1', '1345-2', '1345-3', '1346', '1346-1', '1346-2', '1346-3', '1346-4', '1346-5', '1347', '1347-1', '1347-2', '1347-3', '1347-4', '1347-5', '1347-6', '1347-7', '1348', '1348-1', '1348-2', '1349', '1349-1', '1350', '1350-1', '1350-2', '1351', '1351-1', '1352', '1352-1', '1352-2', '1352-3', '1352-4', '1352-5', '1352-6', '1352-7', '1352-8', '1352-9', '1353', '1354', '1355', '1356', '1357', '1358', '1359', '1360', '1361', '1362', '1363', '1364', '1365', '1366', '1367', '1368', '1369', '1370', '1371', '1372', '1373', '1374', '1375', '1376', '1377', '1378', '1378-1', '1378-2', '1379', '1380', '1381', '1382', '1383', '1383-1', '1383-2', '1384', '1385', '1385-1', '1385-2', '1385-3', '1385-4', '1386', '1386-1', '1387', '1387-1', '1388', '1389', '1390', '1391', '1392', '1393', '1394', '1395', '1396', '1397', '1397-1', '1397-2', '1397-3', '1397-4', '1397-5', '1397-6', '1398', '1399', '1399-1', '1399-2', '1399-3', '1399-4', '1399-5', '1399-6', '1400', '1401', '1402', '1403', '1404', '1405', '1406', '1407', '1408', '1409', '1410', '1411', '1412', '1413', '1414', '1415', '1416', '1417', '1418', '1421', '1422', '1423', '1424', '1425', '1426', '1427', '1428', '1429', '1431', '1432', '1433', '1434', '1435', '1436', '1437', '1438', '1439', '1440', '1441', '1442', '1443', '1444', '1445', '1446', '1447', '1448', '1449', '1451', '1467', '1468', '1469', '1470', '1471', '1472', '1473', '1474', '1475', '1476', '1477', '1478', '1479', '1480', '1482', '1483', '1484', '1485', '1486', '1487', '1488', '1489', '1490', '1491', '1497', '1498', '1499', '1500', '1501', '1503', '1511', '1512', '1513', '1514', '1515', '1516', '1518', '1519', '1520', '1521', '1524', '1525', '1526', '1527', '1536', '1537', '1538', '1539', '1540', '1541', '1542', '1543', '1569', '1570', '1571', '1572', '1573', '1574', '1575', '1576', '1577', '1578', '1579', '1580', '1581', '1582', '1583', '1584', '1585', '1586', '1587', '1588', '1589', '1589-1', '1589-2', '1590', '1591', '1592', '1593', '1594', '1596', '1597', '1598', '1599', '1601', '1601-1', '1601-2', '1601-3', '1601-4', '1602', '1603', '1604', '1605', '1606', '1607', '1608', '1609', '1610', '1611', '1612', '1613', '1614', '1615', '1616', '1617', '1618', '1619', '1620', '1621', '1622', '1623', '1624', '1625', '1626', '1627', '1628', '1629', '1630', '1631', '1632', '1633', '1634', '1635', '1636', '1637', '1638', '1639', '1640', '1641', '1642', '1642-1', '1643', '1644', '1645', '1646', '1646-1', '1647', '1648', '1649', '1650', '1651', '1652', '1653', '1654', '1655', '1656', '1657', '1658', '1659', '1660', '1661', '1662', '1663', '1664', '1665', '1666', '1667', '1668', '1669', '1670', '1671', '1672', '1673', '1674', '1675', '1676', '1677', '1678', '1679', '1680', '1681', '1682', '1683', '1684', '1685', '1686', '1687', '1688', '1689', '1690', '1691', '1693', '1696', '1697', '1698', '1699', '1700', '1701', '1701-1', '1702', '1703', '1704', '1705', '1706', '1707', '1708', '1709', '1710', '1711', '1712', '1713', '1714', '1715', '1716', '1717', '1718', '1719', '1720', '1721', '1722', '1723', '1724', '1725', '1726', '1727', '1728', '1729', '1730', '1731', '1732', '1733', '1734', '1735', '1736', '1737', '1738', '1739', '1740', '1741', '1742', '1743', '1744', '1745', '1746', '1747', '1748', '1749', '1750', '1751', '1751-1', '1752', '1753', '1754', '1755', '1756', '1757', '1758', '1759', '1760', '1761', '1762', '1764', '1765', '1766', '1767', '1768', '1769', '1770', '1771', '1772', '1773', '1774', '1775', '1777', '1778', '1779', '1780', '1782', '1783', '1784', '1785', '1786', '1787', '1788', '1789', '1790', '1791', '1792', '1792-1', '1792-2', '1792-3', '1792-4', '1792-5', '1792-6', '1792-7', '1793', '1794', '1795', '1796', '1797', '1798', '1799', '1799-1', '1800', '1801', '1802', '1803', '1804', '1805', '1806', '1807', '1808', '1809', '1810', '1811', '1812', '1813', '1814', '1815', '1816', '1817', '1818', '1819', '1820', '1821', '1822', '1823', '1824', '1825', '1826', '1827', '1828', '1829', '1830', '1831', '1831-1', '1831-2', '1831-3', '1831-4', '1831-5', '1832', '1832-1', '1832-2', '1833', '1834', '1835', '1836', '1837', '1838', '1839', '1840', '1842', '1843', '1843-1', '1843-2', '1843-3', '1843-4', '1843-5', '1844', '1844-1', '1844-10', '1844-11', '1844-12', '1844-13', '1844-14', '1844-15', '1844-16', '1844-17', '1844-3', '1844-4', '1844-5', '1844-6', '1844-7', '1844-8', '1844-9', '1845', '1845-1', '1846', '1846-1', '1846-2', '1847', '1848', '1849', '1850', '1851', '1852', '1853', '1854', '1854-1', '1855', '1856', '1857', '1858', '1859', '1860', '1861', '1862', '1863', '1864', '1865', '1866', '1867', '1868', '1869', '1870', '1870-1', '1871', '1871-1', '1872', '1872-1', '1872-2', '1873', '1873-1', '1873-10', '1873-11', '1873-12', '1873-13', '1873-14', '1873-15', '1873-16', '1873-17', '1873-18', '1873-2', '1873-3', '1873-4', '1873-5', '1873-6', '1873-7', '1873-8', '1873-9', '1874', '1875', '1876', '1877', '1878', '1879', '1880', '1881', '1882', '1883', '1884', '1885', '1886', '1887', '1888', '1889', '1890', '1891', '1892', '1893', '1894', '1895', '1896', '1897', '1898', '1899', '1900', '1901', '1902', '1903', '1904', '1905', '1906', '1907', '1908', '1909', '1910', '1911', '1912', '1913', '1914', '1915', '1916', '1917', '1918', '1919', '1920', '1921', '1922', '1924', '1925', '1926', '1927', '1928', '1929', '1930', '1931', '1932', '1933', '1934', '1935', '1936', '1937', '1938', '1939', '1940', '1941', '1942', '1943', '1944', '1945', '1946', '1947', '1948', '1949', '1950', '1951', '1952', '1953', '1954', '1955', '1956', '1957', '1958', '1959', '1960', '1961', '1962', '1963', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2015', '2016', '2017', '2018', '2018-1', '2018-2', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026', '2027', '2028', '2029', '2030', '2044', '2045', '2046', '2048', '2049', '2050', '2051', '2052', '2059', '2060', '2061', '2062', '2063', '2064', '2065', '2066', '2067', '2068', '2219', '2220', '2221', '2222', '2223', '2224', '2225', '2226', '2226-1', '2227', '2228', '2229', '2230', '2231', '2232', '2233', '2234', '2235', '2236', '2237', '2238', '2239', '2240', '2241', '2242', '2243', '2244', '2245', '2246', '2247', '2248', '2249', '2250', '2251', '2252', '2253', '2254', '2255', '2256', '2257', '2258', '2259', '2260', '2261', '2262', '2263', '2264', '2265', '2266', '2267', '2268', '2269', '2270', '2271', '2272', '2273', '2274', '2275', '2276', '2277', '2278', '2284', '2285', '2286', '2287', '2287-1', '2288', '2289', '2290', '2291', '2291-1', '2292', '2293', '2294', '2295', '2296', '2297', '2298', '2299', '2300', '2301', '2302', '2303', '2304', '2305', '2305-1', '2306', '2306-1', '2306-2', '2307', '2308', '2309', '2310', '2311', '2312', '2313', '2314', '2315', '2316', '2317', '2318', '2319', '2320', '2321', '2322', '2323', '2324', '2325', '2326', '2329', '2330', '2331', '2331-1', '2332', '2332-1', '2332-2', '2332-3', '2332-4', '2333', '2334', '2335', '2336', '2337', '2338', '2339', '2340', '2341', '2342', '2342-1', '2343', '2344', '2345', '2346', '2347', '2348', '2349', '2350', '2355', '2356', '2358', '2359', '2360', '2361', '2361-1', '2362', '2363', '2363-1', '2364', '2365', '2366', '2367', '2368', '2369', '2370', '2371', '2372', '2372-1', '2372-2', '2372-3', '2372-4', '2372-5', '2373', '2373-1', '2373-2', '2373-3', '2374', '2374-1', '2374-2', '2374-3', '2374-4', '2374-5', '2374-6', '2375', '2376', '2377', '2378', '2379', '2380', '2381', '2382', '2383', '2384', '2385', '2386', '2387', '2388', '2389', '2390', '2391', '2392', '2393', '2394', '2395', '2396', '2397', '2398', '2399', '2400', '2401', '2402', '2403', '2404', '2405', '2406', '2407', '2408', '2409', '2410', '2411', '2412', '2413', '2414', '2415', '2416', '2417', '2418', '2419', '2420', '2421', '2422', '2423', '2424', '2425', '2426', '2427', '2428', '2429', '2430', '2431', '2432', '2433', '2434', '2435', '2436', '2437', '2438', '2439', '2440', '2441', '2442', '2443', '2444', '2445', '2446', '2447', '2448', '2449', '2450', '2451', '2452', '2453', '2454', '2455', '2456', '2457', '2458', '2459', '2460', '2461', '2462', '2463', '2464', '2465', '2466', '2467', '2468', '2469', '2470', '2471', '2472', '2473', '2474', '2488-1', '2488-10', '2488-11', '2488-12', '2488-2', '2488-3', '2488-4', '2488-5', '2488-6', '2488-7', '2488-8', '2488-9', '2489', '2490', '2491', '2492', '2493', '2494', '2495', '2500', '2501', '2502', '2503', '2505', '2507', '2508', '2509', '2510', '2511', '2512', '2513', '2514', '2515', '2516', '2517', '2518', '2519', '2520', '2521', '2522', '2523', '2524', '2525', '2526', '2527', '2528', '2529', '2530', '2531', '2532', '2534']

texts = fetch_articles_texts(articles)   # resumes from artifacts/reference/progression.json
texts = {num: txt for num, txt in texts.items() if txt is not None}

os.makedirs("DATA/inputs", exist_ok=True)
with open("DATA/inputs/civil_code_articles.json", "w", encoding="utf-8") as f:
    json.dump(texts, f, ensure_ascii=False, indent=2)
print(f"{len(texts):,} Civil Code articles saved to DATA/inputs/civil_code_articles.json")

## Extract article numbers

Extracts every article number from the plain-text Civil Code (`DATA/inputs/civil_code_article_list.txt`) with a regex, de-duplicated and sorted numerically.

In [ ]:
import re

# 1. Read the text
with open('DATA/inputs/civil_code_article_list.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# 2. Find all article numbers
pattern = r'Article\s+(\d+(?:-\d+)?)'
matches = re.findall(pattern, text)

# 3. Remove duplicates and sort
unique_articles = sorted(set(matches), key=lambda x: (int(x.split('-')[0]), x))

print(f"{len(unique_articles)} unique articles found")

## Build 2016 old/new equivalences

Processes the official JO 11/02/2016 correspondence table (old vs. new article references) into the shipped `equivalences_2016.csv` and the `equivalences_new_to_old.json` mapping, with a few manual corrections.

In [ ]:
import re
import pandas as pd

# Load Excel file, skipping the first 7 irrelevant rows
df = pd.read_excel("artifacts/reference/equivalences_2016_source.xlsx", sheet_name='_004MH_Code civil Table des art', skiprows=7)  # not shipped — regenerated by this step (see DATA.md)

# Rename columns for clarity
df.columns = ['ancienne_reference', 'nouvelle_reference']

# Extract a simple article number (e.g., 1382 or 1382-1)
def extract_article(text):
    if not isinstance(text, str):
        return None
    match = re.search(r'\b(\d{3,4}(?:-\d+)?)\b', text)
    if match:
        return match.group(1)
    return None

# Apply extraction to both columns
df['ancien_article'] = df['ancienne_reference'].apply(extract_article)
df['nouvel_article'] = df['nouvelle_reference'].apply(extract_article)

# Keep only rows with both old and new article numbers
df_equivalences = df.dropna(subset=['ancien_article', 'nouvel_article'])

# Export to CSV
df_equivalences[['ancien_article', 'nouvel_article']].to_csv(
    "DATA/inputs/equivalences_2016.csv",
    index=False
)

# Create dictionary {old_article: new_article}
dict_equivalences = dict(zip(df_equivalences['ancien_article'], df_equivalences['nouvel_article']))

# Create inverse dictionary {new_article: old_article}
dict_inverse = {v: k for k, v in dict_equivalences.items()}

In [ ]:
# 1. Remove old mapping if it exists
dict_inverse.pop('1144', None)
dict_inverse.pop('1103', None)

# 2. Add corrected mappings
dict_inverse['1144'] = '1304'
dict_inverse['1103'] = '1134'
dict_inverse['1346-1'] = '1250'
dict_inverse['1231-6'] = '1153'

In [ ]:
import json

# Export dictionary: new article -> old article
with open("DATA/inputs/equivalences_new_to_old.json", "w", encoding="utf-8") as f:
    json.dump(dict_inverse, f, ensure_ascii=False, indent=2)